In [6]:
def shockley(V, Is, a):
    return Is * (np.exp(a * V) - 1)
def shockley_odr(B, V): # extra formula voor de odr functie 
    Is, a = B
    return Is * (np.exp(a * V) - 1)
# Deze functie zorgt ervoor dat we de juiste startwaarde vinden voor de fit
# van de shockley functie
def lijn(x, c, d):
    return c * x + d

R = 10000
R2 = 470
R3 = 10
n = 2.0
sn = 0.1

temp_avg = np.array([24.904, 24.952, 24.956, 24.954, 24.933,
                     24.946, 24.972, 24.980, 24.954, 24.986])
# De temperatuur in Kelvin
T  = np.mean(temp_avg) + 273.15
sT = 1.0

USB = np.array([6.02, 5.77, 5.52, 5.26, 5.02, 4.74, 4.54, 4.24, 3.99, 3.74, 3.49, 3.24,
                2.99, 2.74, 2.49, 2.24, 1.99, 1.74, 1.49, 1.24, 0.99, 0.74, 0.49, 0.24])
VR1 = np.array([0.498, 0.379, 0.342, 0.295, 0.261, 0.266, 0.216, 0.208, 0.161, 0.129,
                0.119, 0.105, 0.090, 0.072, 0.061, 0.048, 0.039, 0.033, 0.025, 0.019,
                0.014, 0.009, 0.006, 0.002]) / 1e3
kB_doel = 1.380649e-23
a_doel  = q / (kB_doel * 2 * T)  # De exacte theoretische helling (~19.46 1/V)


I_theorie   = 1.49e-9 * (np.exp(a_doel * U) - 1)
VR1_theorie = I_theorie * R 


VR1_hypothetisch = (VR1_theorie + np.random.normal(0, sVR1))*1e3

delta_USB = 0.0050 * USB + 0.01
sUSB      = delta_USB / np.sqrt(3)   # berekenen van de onzerkerheid per punt door specificaties van Dynatek en Thurlby 1905a 
delta_VR1 = 0.00015 * VR1_hypothetisch + (0.0003/1e3)
sVR1      = delta_VR1 / np.sqrt(3)
I_D = VR1_hypothetisch / R
sI_D = sVR1 / R
factor = R3 / (R2 + R3)
U    = factor * USB
U_D = U - VR1_hypothetisch
sU_D = np.sqrt((factor * sUSB)**2 + sVR1**2)


# Voor het vinden van die startwaarde gebruiken we een fit met een rechte lijn, wat we in principe verwachten van de grafiek.
# We willen alleen punten die boven de 0.03 uitkomen, I_d > 0 omdat je niet de log van <= 0 kan nemen.
# p0 rekent de parameters om naar waarden die passen bij de shockley functie

mask = (U_D > 0.03) & (I_D > 0)
mask = (U_D > 0.03) & (I_D > sI_D) # extra mask voor de onzerkerheden

(c, d), _ = curve_fit(lijn, U_D[mask], np.log10(I_D[mask])) # uitvoeren van curvefit
p0 = [10 ** d, c / np.log10(np.e)]

model = odr.Model(shockley_odr) # het gebruiken van odr pakket voor onzerkerheid
data  = odr.RealData(U_D, I_D, sx=sU_D, sy=sI_D)
myodr = odr.ODR(data, model, beta0=p0)
output = myodr.run()

Is, a = output.beta
sIs, sa = output.sd_beta  # Standaardfouten op de fitparameters

# --- 6. Constante van Boltzmann met onzekerheid ---
kB  = q / (a * n * T)
skB = kB * np.sqrt((sa / a)**2 + (sn / n)**2 + (sT / T)**2)

# Constante van Boltzmann met onzekerheid
kB  = q / (a * n * T)
skB = kB * np.sqrt((sa / a)**2 + (sn / n)**2 + (sT / T)**2)


V = np.linspace(U_D.min(), U_D.max(), 500)


#printen van uitgekomen resultaten
print(f"Is  = {Is:.3e} ± {sIs:.1e} A")
print(f"a   = {a:.3g} ± {sa:.2g} 1/V")
print(f"k_B = {kB:.3e} ± {skB:.1e} J/K   (literatuur: 1.381e-23)")

#plotten van de functies met errorbars
plt.errorbar(U_D[mask], I_D[mask], xerr=sU_D[mask], yerr=sI_D[mask], fmt='o', capsize=3, label='meting') 
plt.yscale('log')

plt.plot(V, shockley(V, *output.beta), label='ODR fit')
plt.plot(U_D[~mask], I_D[~mask], 'o', color='lightgray', label='buiten fitbereik')

plt.xlabel('$U_D$ (V)')
plt.ylabel('$I_D$ (A)')
plt.legend()
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.show()



NameError: name 'U' is not defined